In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="15gJq9gJtEMdlM4IB5mO6_2--y73DK9dg", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import torch
import torch.nn as nn
import torch.distributed as dist
import sys
import os
import math
import matplotlib.pyplot as plt
import numpy as np

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. We will simulate multi-GPU with CPU tensors.")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

%matplotlib inline

# ZeRO from Scratch -- Vizuara

In this notebook, we will implement simplified versions of ZeRO Stage 1 and Stage 2 in pure PyTorch. We will simulate multiple GPUs on a single device, so you can see exactly how optimizer state partitioning, reduce-scatter, and allgather work at the code level.

**What you will build:** A working ZeRO-1 and ZeRO-2 optimizer that partitions states across simulated GPU ranks, trains a small model, and produces the same results as standard data parallelism.

In [ ]:
#@title 🎧 Listen: Simulating Gpus Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_simulating_gpus_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Simulating Multiple GPUs

Real ZeRO uses NCCL collectives across multiple GPUs. Since we are in a single-GPU Colab environment, we will simulate N "ranks" by keeping separate copies of what each GPU would store. The key operations we need to simulate are:

- **AllReduce**: average a tensor across all ranks (each rank gets the full average)
- **ReduceScatter**: average a tensor and scatter chunks so each rank gets 1/N
- **AllGather**: each rank contributes its chunk, all ranks get the full tensor

This simulation captures the semantics perfectly, just without the actual GPU-to-GPU communication.

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Comm Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_simulated_comm_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimulatedComm:
    """
    Simulates distributed communication primitives.
    Instead of actual GPU-to-GPU transfers, we operate on
    lists of tensors (one per simulated rank).
    """

    def __init__(self, world_size):
        self.world_size = world_size

    def allreduce(self, tensors):
        """
        AllReduce: average tensors across all ranks.
        Input: list of N tensors (one per rank)
        Output: list of N tensors, each containing the average.
        """
        avg = sum(tensors) / self.world_size
        return [avg.clone() for _ in range(self.world_size)]

    def reduce_scatter(self, tensors):
        """
        ReduceScatter: average tensors, then scatter 1/N to each rank.
        Input: list of N tensors (one per rank), each of size P
        Output: list of N tensors, each of size P/N (the rank's partition)
        """
        avg = sum(tensors) / self.world_size
        chunk_size = avg.numel() // self.world_size
        chunks = [avg[i * chunk_size:(i + 1) * chunk_size].clone()
                  for i in range(self.world_size)]
        return chunks

    def allgather(self, chunks):
        """
        AllGather: each rank contributes its chunk, all get the full tensor.
        Input: list of N tensors (chunks, one per rank)
        Output: list of N tensors, each being the concatenation of all chunks.
        """
        full = torch.cat(chunks)
        return [full.clone() for _ in range(self.world_size)]


# Test the communication primitives
comm = SimulatedComm(world_size=4)

# Test allreduce
tensors = [torch.tensor([1.0, 2.0, 3.0, 4.0]) * (i + 1) for i in range(4)]
print("Before AllReduce:")
for i, t in enumerate(tensors):
    print(f"  Rank {i}: {t.tolist()}")

result = comm.allreduce(tensors)
print("After AllReduce:")
for i, t in enumerate(result):
    print(f"  Rank {i}: {t.tolist()}")

# Test reduce_scatter
rs_result = comm.reduce_scatter(tensors)
print("\nAfter ReduceScatter:")
for i, t in enumerate(rs_result):
    print(f"  Rank {i}: {t.tolist()} (only its partition)")

# Test allgather
ag_result = comm.allgather(rs_result)
print("\nAfter AllGather:")
for i, t in enumerate(ag_result):
    print(f"  Rank {i}: {t.tolist()} (full tensor reconstructed)")

In [ ]:
#@title 🎧 Narration: Think About This Comm
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_think_about_this_comm.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Think About This

Notice that ReduceScatter + AllGather together give you the same result as AllReduce. This is not a coincidence -- AllReduce can always be decomposed into a ReduceScatter followed by an AllGather. ZeRO Stage 2 exploits this: instead of doing a full AllReduce on gradients (which gives every GPU the complete average), it does only the ReduceScatter (so each GPU keeps just its partition), and saves the AllGather for later (to broadcast the updated weights).

In [ ]:
#@title 🎧 Transition: Simple Model Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_simple_model_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. A Simple Model for Testing

We will create a small linear model so we can verify correctness by comparing ZeRO training to standard training.

In [ ]:
#@title 🎧 Code Walkthrough: Model Helpers
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_model_helpers.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def create_model(input_dim=32, hidden_dim=64, output_dim=8):
    """Create a simple MLP for testing."""
    model = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim),
    )
    return model


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def flatten_params(model):
    """Flatten all model parameters into a single 1D tensor."""
    return torch.cat([p.data.view(-1) for p in model.parameters()])


def flatten_grads(model):
    """Flatten all gradients into a single 1D tensor."""
    grads = []
    for p in model.parameters():
        if p.grad is not None:
            grads.append(p.grad.view(-1))
        else:
            grads.append(torch.zeros_like(p.data.view(-1)))
    return torch.cat(grads)


def unflatten_to_model(flat_tensor, model):
    """Copy a flat tensor back into model parameters."""
    offset = 0
    for p in model.parameters():
        numel = p.numel()
        p.data.copy_(flat_tensor[offset:offset + numel].view(p.shape))
        offset += numel


model = create_model()
total_params = count_params(model)
print(f"Model parameters: {total_params:,}")
print(f"Parameter shapes: {[p.shape for p in model.parameters()]}")

In [ ]:
#@title 🎧 Transition: Standard Dp Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_standard_dp_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Standard Data Parallelism (Baseline)

First, let us implement standard data parallelism as a baseline. This is what ZeRO optimizes.

In [ ]:
#@title 🎧 Code Walkthrough: Train Standard Dp
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_train_standard_dp.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def train_standard_dp(model_template, data_shards, n_steps, lr=0.01, world_size=4):
    """
    Standard Data Parallelism training.

    Each rank has a full copy of the model, computes forward/backward on its
    data shard, then all-reduces the gradients.
    """
    comm = SimulatedComm(world_size)

    # Each rank gets a copy of the model
    import copy
    models = [copy.deepcopy(model_template) for _ in range(world_size)]
    optimizers = [torch.optim.Adam(m.parameters(), lr=lr) for m in models]

    losses_per_step = []

    for step in range(n_steps):
        step_losses = []

        # Forward + backward on each rank
        for rank in range(world_size):
            optimizers[rank].zero_grad()
            x, y = data_shards[rank][step % len(data_shards[rank])]
            pred = models[rank](x)
            loss = nn.functional.mse_loss(pred, y)
            loss.backward()
            step_losses.append(loss.item())

        # AllReduce gradients
        all_grads = [flatten_grads(m) for m in models]
        avg_grads = comm.allreduce(all_grads)

        # Write averaged gradients back and step
        for rank in range(world_size):
            offset = 0
            for p in models[rank].parameters():
                numel = p.numel()
                p.grad.copy_(avg_grads[rank][offset:offset + numel].view(p.shape))
                offset += numel
            optimizers[rank].step()

        losses_per_step.append(np.mean(step_losses))

    # All models should be identical
    return models[0], losses_per_step


# Generate synthetic data
torch.manual_seed(42)
world_size = 4
n_steps = 100

data_shards = []
for rank in range(world_size):
    shard = []
    for _ in range(50):
        x = torch.randn(16, 32)  # batch=16, input=32
        y = torch.randn(16, 8)   # batch=16, output=8
        shard.append((x, y))
    data_shards.append(shard)

# Train baseline
torch.manual_seed(42)
model_template = create_model()
baseline_model, baseline_losses = train_standard_dp(
    model_template, data_shards, n_steps, lr=0.001, world_size=world_size
)

print(f"Baseline training complete. Final loss: {baseline_losses[-1]:.4f}")
baseline_params = flatten_params(baseline_model)

In [ ]:
#@title 🎧 Transition: Zero1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_zero1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. ZeRO Stage 1: Partitioned Optimizer States

In ZeRO-1, each rank stores only 1/N of the optimizer states. The forward and backward passes are identical to standard DP. The key changes are:

1. After AllReduce of gradients, each rank only updates its 1/N partition
2. After the optimizer step, an AllGather broadcasts the updated weight partitions

Let us implement this.

In [ ]:
#@title 🎧 Code Walkthrough: Zero1 Optimizer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_zero1_optimizer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class ZeROStage1Optimizer:
    """
    Simplified ZeRO Stage 1: partition Adam optimizer states across ranks.

    Each rank stores master weights (fp32), m, and v for only its 1/N partition.
    """

    def __init__(self, flat_params, world_size, rank, lr=0.001, betas=(0.9, 0.999), eps=1e-8):
        self.world_size = world_size
        self.rank = rank
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.step_count = 0

        total_params = flat_params.numel()
        self.chunk_size = total_params // world_size

        # Each rank only stores optimizer states for its partition
        start = rank * self.chunk_size
        end = start + self.chunk_size

        # Master weights (fp32 copy of the partition)
        self.master_weights = flat_params[start:end].float().clone()
        # First moment (m) - initialized to zero
        self.m = torch.zeros_like(self.master_weights)
        # Second moment (v) - initialized to zero
        self.v = torch.zeros_like(self.master_weights)

    def step(self, grad_partition):
        """
        Perform Adam update on this rank's partition.

        Args:
            grad_partition: the 1/N slice of the full averaged gradient
                            corresponding to this rank

        Returns:
            updated_partition: the updated fp16 weight partition (for allgather)
        """
        self.step_count += 1
        t = self.step_count

        # Adam update on the partition
        g = grad_partition.float()
        self.m = self.beta1 * self.m + (1 - self.beta1) * g
        self.v = self.beta2 * self.v + (1 - self.beta2) * g ** 2

        # Bias correction
        m_hat = self.m / (1 - self.beta1 ** t)
        v_hat = self.v / (1 - self.beta2 ** t)

        # Update master weights
        self.master_weights -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

        return self.master_weights.clone()

    def memory_usage(self):
        """Return memory used by this rank's optimizer states in bytes."""
        # master_weights + m + v, all fp32 (4 bytes each)
        return self.chunk_size * 4 * 3  # 12 bytes per param in partition


# Quick test
flat = torch.randn(1000)
opt = ZeROStage1Optimizer(flat, world_size=4, rank=0)
print(f"Partition size: {opt.chunk_size} params")
print(f"Optimizer memory for rank 0: {opt.memory_usage()} bytes")
print(f"  (vs full Adam: {1000 * 12} bytes)")
print(f"  Savings: {1 - opt.memory_usage() / (1000 * 12):.0%}")

In [ ]:
#@title 🎧 Code Walkthrough: Train Zero Stage1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_train_zero_stage1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def train_zero_stage1(model_template, data_shards, n_steps, lr=0.001, world_size=4):
    """
    ZeRO Stage 1 training: partition optimizer states, allreduce gradients,
    each rank updates its partition, allgather updated weights.
    """
    comm = SimulatedComm(world_size)
    import copy

    # Each rank has a full copy of the model (for forward/backward)
    models = [copy.deepcopy(model_template) for _ in range(world_size)]

    # But each rank only has optimizer states for its partition
    flat_init = flatten_params(models[0])
    optimizers = [
        ZeROStage1Optimizer(flat_init, world_size, rank, lr=lr)
        for rank in range(world_size)
    ]

    losses_per_step = []
    chunk_size = flat_init.numel() // world_size

    for step in range(n_steps):
        step_losses = []

        # Forward + backward on each rank
        for rank in range(world_size):
            for p in models[rank].parameters():
                if p.grad is not None:
                    p.grad.zero_()
            x, y = data_shards[rank][step % len(data_shards[rank])]
            pred = models[rank](x)
            loss = nn.functional.mse_loss(pred, y)
            loss.backward()
            step_losses.append(loss.item())

        # AllReduce gradients (same as standard DP)
        all_grads = [flatten_grads(m) for m in models]
        avg_grads = comm.allreduce(all_grads)

        # Each rank updates ONLY its partition
        updated_partitions = []
        for rank in range(world_size):
            start = rank * chunk_size
            end = start + chunk_size
            grad_partition = avg_grads[rank][start:end]
            updated = optimizers[rank].step(grad_partition)
            updated_partitions.append(updated)

        # AllGather the updated weight partitions
        full_weights_list = comm.allgather(updated_partitions)

        # Write the full weights back to all models
        for rank in range(world_size):
            unflatten_to_model(full_weights_list[rank], models[rank])

        losses_per_step.append(np.mean(step_losses))

    return models[0], losses_per_step


# Train with ZeRO Stage 1
torch.manual_seed(42)
model_template = create_model()
z1_model, z1_losses = train_zero_stage1(
    model_template, data_shards, n_steps, lr=0.001, world_size=world_size
)

print(f"ZeRO-1 training complete. Final loss: {z1_losses[-1]:.4f}")

# Compare final parameters to baseline
z1_params = flatten_params(z1_model)
diff = (z1_params - baseline_params).abs().max().item()
print(f"Max parameter difference vs baseline: {diff:.6e}")
if diff < 1e-4:
    print("ZeRO-1 produces the same model as standard DP!")
else:
    print("Small numerical differences (expected due to floating point).")

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn: ZeRO Stage 2

### TODO 1: Implement ZeRO Stage 2 Training Loop

ZeRO Stage 2 differs from Stage 1 in one key way: instead of AllReduce on gradients (which gives every rank the full averaged gradient), it uses **ReduceScatter** (each rank gets only its 1/N partition of the averaged gradient). This saves gradient memory.

The training loop becomes:
1. Forward + backward on each rank (same as before)
2. **ReduceScatter** gradients (NOT AllReduce)
3. Each rank updates its partition using the received gradient chunk
4. AllGather updated weights (same as ZeRO-1)

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def train_zero_stage2(model_template, data_shards, n_steps, lr=0.001, world_size=4):
    """
    ZeRO Stage 2: partition optimizer states AND gradients.
    Uses reduce-scatter instead of allreduce for gradients.
    """
    comm = SimulatedComm(world_size)
    import copy

    models = [copy.deepcopy(model_template) for _ in range(world_size)]

    flat_init = flatten_params(models[0])
    optimizers = [
        ZeROStage1Optimizer(flat_init, world_size, rank, lr=lr)
        for rank in range(world_size)
    ]

    losses_per_step = []

    for step in range(n_steps):
        step_losses = []

        # Forward + backward on each rank
        for rank in range(world_size):
            for p in models[rank].parameters():
                if p.grad is not None:
                    p.grad.zero_()
            x, y = data_shards[rank][step % len(data_shards[rank])]
            pred = models[rank](x)
            loss = nn.functional.mse_loss(pred, y)
            loss.backward()
            step_losses.append(loss.item())

        # ============ TODO ============
        # Step 1: Collect flat gradients from all ranks
        all_grads = [flatten_grads(m) for m in models]

        # Step 2: ReduceScatter (NOT AllReduce!)
        # This gives each rank only its 1/N partition of the averaged gradient
        grad_partitions = ???  # YOUR CODE: use comm.reduce_scatter

        # Step 3: Each rank updates its partition
        updated_partitions = []
        for rank in range(world_size):
            updated = ???  # YOUR CODE: call optimizers[rank].step with the right partition
            updated_partitions.append(updated)

        # Step 4: AllGather updated weights
        full_weights_list = ???  # YOUR CODE: use comm.allgather
        # ==============================

        # Write the full weights back to all models
        for rank in range(world_size):
            unflatten_to_model(full_weights_list[rank], models[rank])

        losses_per_step.append(np.mean(step_losses))

    return models[0], losses_per_step


# Train with ZeRO Stage 2
torch.manual_seed(42)
model_template = create_model()
z2_model, z2_losses = train_zero_stage2(
    model_template, data_shards, n_steps, lr=0.001, world_size=world_size
)

print(f"ZeRO-2 training complete. Final loss: {z2_losses[-1]:.4f}")

In [ ]:
#@title 🎧 Before You Start: Todo1 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 1
z2_params = flatten_params(z2_model)
diff_z2 = (z2_params - baseline_params).abs().max().item()
print(f"Max parameter difference (ZeRO-2 vs baseline): {diff_z2:.6e}")
assert diff_z2 < 1e-3, f"ZeRO-2 should match baseline. Diff: {diff_z2:.6e}"
print("TODO 1 passed! ZeRO-2 matches standard DP.")

In [ ]:
#@title 🎧 Transition: Loss Curves Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_loss_curves_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Comparing Loss Curves

All three approaches should produce nearly identical loss curves, since they compute the same mathematical operations -- just with different memory layouts.

In [ ]:
#@title 🎧 What to Look For: Loss Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_loss_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline_losses, label='Standard DP', linewidth=2, color='#F44336')
ax.plot(z1_losses, label='ZeRO-1', linewidth=2, linestyle='--', color='#FF9800')
ax.plot(z2_losses, label='ZeRO-2', linewidth=2, linestyle=':', color='#2196F3')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss: Standard DP vs ZeRO-1 vs ZeRO-2', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Transition: Memory Accounting Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_memory_accounting_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Memory Accounting

Let us compare the actual memory used by each rank across the approaches.

In [ ]:
#@title 🎧 What to Look For: Memory Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_memory_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
total_params = count_params(model_template)
print(f"Model parameters: {total_params:,}\n")

# Standard DP: each rank stores full weights + gradients + optimizer
dp_memory = {
    'weights': total_params * 4,     # fp32
    'gradients': total_params * 4,   # fp32
    'optimizer': total_params * 4 * 2,  # m + v (fp32 each)
}

# ZeRO-1: partitioned optimizer, full weights + gradients
z1_memory = {
    'weights': total_params * 4,
    'gradients': total_params * 4,
    'optimizer': (total_params // world_size) * 4 * 3,  # master + m + v partitioned
}

# ZeRO-2: partitioned optimizer + gradients, full weights
z2_memory = {
    'weights': total_params * 4,
    'gradients': (total_params // world_size) * 4,  # only during optimizer step
    'optimizer': (total_params // world_size) * 4 * 3,
}

for name, mem in [('Standard DP', dp_memory), ('ZeRO-1', z1_memory), ('ZeRO-2', z2_memory)]:
    total = sum(mem.values())
    print(f"{name}:")
    for component, bytes_val in mem.items():
        print(f"  {component:12s}: {bytes_val:8,} bytes ({bytes_val/total*100:5.1f}%)")
    print(f"  {'TOTAL':12s}: {total:8,} bytes\n")

# Bar chart
labels = ['Standard DP', 'ZeRO-1', 'ZeRO-2']
w = [dp_memory['weights'], z1_memory['weights'], z2_memory['weights']]
g = [dp_memory['gradients'], z1_memory['gradients'], z2_memory['gradients']]
o = [dp_memory['optimizer'], z1_memory['optimizer'], z2_memory['optimizer']]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
ax.bar(x, w, 0.4, label='Weights', color='#2196F3')
ax.bar(x, g, 0.4, bottom=w, label='Gradients', color='#4CAF50')
ax.bar(x, o, 0.4, bottom=[wi+gi for wi, gi in zip(w, g)], label='Optimizer', color='#9C27B0')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Memory per Rank (bytes)')
ax.set_title('Per-Rank Memory: Simulated Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Your Turn: Communication Volume Analysis

### TODO 2: Count Communication Bytes

Instrument the training functions to count the total bytes communicated. For each collective operation, count the bytes sent and received per rank.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class CountingComm(SimulatedComm):
    """
    Communication simulator that also counts total bytes transferred.
    """

    def __init__(self, world_size):
        super().__init__(world_size)
        self.total_bytes = 0

    def allreduce(self, tensors):
        # ============ TODO ============
        # AllReduce transfers approximately 2 * data_size bytes per rank
        # (reduce-scatter + allgather internally)
        # Count bytes for ONE rank.
        data_size = tensors[0].numel() * tensors[0].element_size()
        self.total_bytes += ???  # YOUR CODE: how many bytes does allreduce transfer per rank?
        # ==============================
        return super().allreduce(tensors)

    def reduce_scatter(self, tensors):
        # ============ TODO ============
        # ReduceScatter transfers approximately data_size bytes per rank
        data_size = tensors[0].numel() * tensors[0].element_size()
        self.total_bytes += ???  # YOUR CODE: how many bytes?
        # ==============================
        return super().reduce_scatter(tensors)

    def allgather(self, chunks):
        # ============ TODO ============
        # AllGather transfers approximately (N-1)/N * data_size bytes per rank
        # Simplified: count full data_size
        total_size = sum(c.numel() * c.element_size() for c in chunks)
        self.total_bytes += ???  # YOUR CODE: how many bytes?
        # ==============================
        return super().allgather(chunks)


# Test: count bytes for one step of each approach
print("Communication bytes per step per rank:")
print(f"  Total parameters: {total_params:,}")
print(f"  Bytes per param (fp32): 4\n")

# Standard DP: allreduce gradients
cc_dp = CountingComm(world_size)
dummy_grads = [torch.randn(total_params) for _ in range(world_size)]
cc_dp.allreduce(dummy_grads)
print(f"  Standard DP: {cc_dp.total_bytes:,} bytes")

# ZeRO-1: allreduce gradients + allgather weights
cc_z1 = CountingComm(world_size)
cc_z1.allreduce(dummy_grads)
dummy_chunks = [torch.randn(total_params // world_size) for _ in range(world_size)]
cc_z1.allgather(dummy_chunks)
print(f"  ZeRO-1:      {cc_z1.total_bytes:,} bytes")

# ZeRO-2: reduce_scatter gradients + allgather weights
cc_z2 = CountingComm(world_size)
cc_z2.reduce_scatter(dummy_grads)
cc_z2.allgather(dummy_chunks)
print(f"  ZeRO-2:      {cc_z2.total_bytes:,} bytes")

In [ ]:
#@title 🎧 Before You Start: Todo2 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_todo2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 2
# Standard DP allreduce should be ~2 * data_size
expected_dp = 2 * total_params * 4
assert abs(cc_dp.total_bytes - expected_dp) < total_params, (
    f"DP comm should be ~{expected_dp}, got {cc_dp.total_bytes}")
# ZeRO-2 should have same total as ZeRO-1 and DP
# (reduce_scatter + allgather = allreduce)
print(f"DP comm:     {cc_dp.total_bytes:,} bytes")
print(f"ZeRO-2 comm: {cc_z2.total_bytes:,} bytes")
print("TODO 2 passed!")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Implement Gradient Accumulation with ZeRO-2

In practice, ZeRO is often combined with gradient accumulation to increase the effective batch size. Implement a version of ZeRO-2 that accumulates gradients over multiple micro-batches before the reduce-scatter.

In [ ]:
#@title 🎧 Before You Start: Todo3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_todo3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def train_zero2_with_accumulation(
    model_template, data_shards, n_steps, lr=0.001,
    world_size=4, accum_steps=4
):
    """
    ZeRO Stage 2 with gradient accumulation.

    Each rank accumulates gradients over `accum_steps` micro-batches
    before communicating.
    """
    comm = SimulatedComm(world_size)
    import copy

    models = [copy.deepcopy(model_template) for _ in range(world_size)]
    flat_init = flatten_params(models[0])
    optimizers = [
        ZeROStage1Optimizer(flat_init, world_size, rank, lr=lr)
        for rank in range(world_size)
    ]

    losses_per_step = []
    data_idx = 0

    for step in range(n_steps):
        # ============ TODO ============
        # Accumulate gradients over accum_steps micro-batches.
        # 1. Zero gradients at the start
        # 2. Loop over accum_steps micro-batches, computing and ACCUMULATING grads
        #    (do NOT zero grads between micro-batches)
        # 3. Divide accumulated gradients by accum_steps (for averaging)
        # 4. Then do reduce-scatter + optimizer step + allgather as before

        step_losses = []

        # Zero gradients
        for rank in range(world_size):
            for p in models[rank].parameters():
                if p.grad is not None:
                    p.grad.zero_()

        # Accumulate over micro-batches
        for micro in range(accum_steps):
            for rank in range(world_size):
                x, y = data_shards[rank][data_idx % len(data_shards[rank])]
                pred = models[rank](x)
                loss = nn.functional.mse_loss(pred, y)
                ???  # YOUR CODE: backward (accumulates gradients)
                step_losses.append(loss.item())
            data_idx += 1

        # Average accumulated gradients
        for rank in range(world_size):
            for p in models[rank].parameters():
                if p.grad is not None:
                    ???  # YOUR CODE: divide gradients by accum_steps

        # ==============================

        # ReduceScatter + optimizer step + AllGather (same as ZeRO-2)
        all_grads = [flatten_grads(m) for m in models]
        grad_partitions = comm.reduce_scatter(all_grads)

        updated_partitions = []
        for rank in range(world_size):
            updated = optimizers[rank].step(grad_partitions[rank])
            updated_partitions.append(updated)

        full_weights_list = comm.allgather(updated_partitions)
        for rank in range(world_size):
            unflatten_to_model(full_weights_list[rank], models[rank])

        losses_per_step.append(np.mean(step_losses))

    return models[0], losses_per_step


# Train with gradient accumulation
torch.manual_seed(42)
model_template = create_model()
z2_accum_model, z2_accum_losses = train_zero2_with_accumulation(
    model_template, data_shards, n_steps=50, lr=0.001,
    world_size=world_size, accum_steps=4
)

print(f"ZeRO-2 + GradAccum training complete. Final loss: {z2_accum_losses[-1]:.4f}")
print(f"Effective batch size: {16 * world_size * 4} = micro_batch * world_size * accum_steps")

In [ ]:
#@title 🎧 Before You Start: Todo3 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_todo3_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 3
assert len(z2_accum_losses) == 50, f"Expected 50 steps, got {len(z2_accum_losses)}"
assert z2_accum_losses[-1] < z2_accum_losses[0], "Loss should decrease over training"
print("TODO 3 passed!")

In [ ]:
#@title 🎧 Transition: Grad Accum Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_grad_accum_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Visualization: Gradient Accumulation Effect

In [ ]:
#@title 🎧 What to Look For: Grad Accum Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_grad_accum_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z2_losses, label='ZeRO-2 (no accum)', linewidth=2, color='#2196F3')
ax.plot(z2_accum_losses, label='ZeRO-2 + GradAccum (4 steps)', linewidth=2, color='#4CAF50')
ax.set_xlabel('Optimizer Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Effect of Gradient Accumulation with ZeRO-2', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. Reflection and Next Steps

### Think About This
1. Why does ZeRO-2 use ReduceScatter instead of AllReduce? What is the concrete memory benefit?
2. In our simulation, the models on all ranks are identical after each step. In real distributed training, could they ever diverge? When?
3. How would you extend this to ZeRO Stage 3? What additional AllGather calls would be needed?

### What Comes Next
In Notebook 3, we will use Microsoft's DeepSpeed library to apply ZeRO to real model training. You will see how a single JSON config file replaces all the manual partitioning we built here.

### Key Takeaway
ZeRO does not change the mathematical computation -- it changes *where* data is stored and *how* it is communicated. The optimizer update is identical; only the memory layout and communication pattern differ. This is why ZeRO produces the exact same model as standard data parallelism.